# 06 - Restart-case closure and canonical case files

Run from `code/02_diagnostics` on STREAM2. Archived MLS inputs below `PAPER1_ARCHIVE_ROOT` are read-only; staged scientific inputs come from `PAPER1_PREPROCESSED_ROOT`; every new product is schema-validated and atomically written below `PAPER1_DERIVED_ROOT` (default: repository-local `Paper1/runtime`; overrides must resolve to that runtime directory or one of its descendants).


## Cross-product closure gate

Inputs: canonical O3, dynamics, EP flux, NAM/AO, and verification products for all three restart cases. Outputs: an atomic canonical-case assembler. Method: require identical 30-member/date coordinates, finite uninterrupted initialization-to-0008-05-31 display windows, natural-month EP, and March's direct uncalibrated EOF projection.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

def discover_diagnostic_directory():
    candidates = (
        Path.cwd(), Path.cwd() / "02_diagnostics",
        Path.cwd() / "Paper1" / "02_diagnostics",
        Path.cwd() / "code_cleaned" / "Paper1" / "02_diagnostics",
    )
    for candidate in candidates:
        if (candidate / "lib" / "workflow_io.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate Paper1/02_diagnostics/lib from the current working directory"
    )

NOTEBOOK_DIR = discover_diagnostic_directory()
LIB = NOTEBOOK_DIR / "lib"
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))

from workflow_io import (
    PRODUCT_VERSION, archive_root, derived_root, preprocessed_root, product_path,
    write_csv_atomic, write_netcdf_atomic,
)

ARCHIVE_ROOT = archive_root()
PREPROCESSED_ROOT = preprocessed_root()
DERIVED_ROOT = derived_root()
MARINA_ROOT = Path(os.environ.get(
    "PAPER1_MARINA_ROOT",
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.002_0008/Mar",
))
OVERWRITE = os.environ.get("PAPER1_OVERWRITE_STAGING", "0") == "1"
print("read-only archive root:", ARCHIVE_ROOT)
print("preprocessed staging input root:", PREPROCESSED_ROOT)
print("read-only Marina March root:", MARINA_ROOT)
print("staging output root:", DERIVED_ROOT)
print("diagnostic notebook directory:", NOTEBOOK_DIR)

CASE_DISPLAY_START = {"0008-01": 80101, "0008-02": 80201, "0008-03": 80301}
CASE_DISPLAY_NT = {"0008-01": 151, "0008-02": 120, "0008-03": 92}
DISPLAY_END_DATE = 80531
NOLEAP_MONTH_LENGTHS = (31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31)

def noleap_date_range(start_date, end_date):
    start_date, end_date = int(start_date), int(end_date)
    start_year, end_year = start_date // 10000, end_date // 10000
    if start_year != end_year:
        raise ValueError("Display-window helper requires one no-leap model year")
    dates = []
    for month, length in enumerate(NOLEAP_MONTH_LENGTHS, start=1):
        for day in range(1, length + 1):
            value = start_year * 10000 + month * 100 + day
            if start_date <= value <= end_date:
                dates.append(value)
    return np.asarray(dates, dtype=int)

def require_case_display_window(case, dates):
    dates = np.asarray(dates, dtype=int)
    if dates.ndim != 1 or np.unique(dates).size != dates.size or np.any(np.diff(dates) <= 0):
        raise RuntimeError(f"{case}: dates must be one-dimensional, unique, and strictly increasing")
    expected = noleap_date_range(CASE_DISPLAY_START[case], DISPLAY_END_DATE)
    if expected.size != CASE_DISPLAY_NT[case]:
        raise AssertionError(f"{case}: internal display-day count is wrong")
    lookup = {int(value): index for index, value in enumerate(dates)}
    missing = [int(value) for value in expected if int(value) not in lookup]
    if missing:
        raise RuntimeError(
            f"{case}: display window is incomplete through 0008-05-31; "
            f"missing {missing[:5]}"
        )
    indices = np.asarray([lookup[int(value)] for value in expected], dtype=int)
    return expected, indices

def require_finite_display_values(context, values, indices, *, time_axis):
    selected = np.take(np.asarray(values, dtype=float), indices, axis=time_axis)
    finite = np.isfinite(selected)
    if not bool(finite.all()):
        raise RuntimeError(
            f"{context}: display window through 0008-05-31 has "
            f"{int(finite.size - finite.sum())} non-finite value(s)"
        )

def assemble_case(case):
    paths = {
        "ozone": product_path("ozone", f"hindcast_{case}_partial_o3.nc"),
        "dynamics": product_path("dynamics", f"hindcast_{case}.nc"),
        "nam": product_path("nam", f"hindcast_{case}_nam_ao.nc"),
        "epflux": product_path("epflux", f"hindcast_{case}_epflux.nc"),
    }
    opened = {name: xr.open_dataset(path, decode_times=False) for name, path in paths.items()}
    verification = xr.open_dataset(product_path("verification", f"{case}_daily.nc"), decode_times=False)
    try:
        dates = np.asarray(opened["ozone"].date.values, dtype=int)
        members = np.asarray(opened["ozone"].member.values).astype(str)
        for name, dataset in opened.items():
            if dataset.sizes.get("member") != 30:
                raise RuntimeError(f"{case} {name}: expected exactly 30 members")
            if not np.array_equal(dates, np.asarray(dataset.date.values, dtype=int)):
                raise RuntimeError(f"{case} {name}: date coordinate differs from ozone")
            if not np.array_equal(members, np.asarray(dataset.member.values).astype(str)):
                raise RuntimeError(f"{case} {name}: member coordinate differs from ozone")
        if not np.array_equal(dates, np.asarray(verification.date.values, dtype=int)):
            raise RuntimeError(f"{case}: verification date coordinate differs from ozone")
        display_dates, display_indices = require_case_display_window(case, dates)
        if (
            verification.attrs.get("display_window_complete") != "True"
            or verification.attrs.get("display_end_date") != "00080531"
            or int(verification.attrs.get("display_day_count", -1)) != display_dates.size
        ):
            raise RuntimeError(f"{case}: verification display-window provenance is invalid")
        nam_plev = np.asarray(opened["nam"].plev.values, dtype=float)
        if not np.array_equal(nam_plev, np.asarray(verification.plev.values, dtype=float)):
            raise RuntimeError(f"{case}: verification plev coordinate differs from NAM")
        if opened["epflux"].attrs.get("natural_month_n2") != "True":
            raise RuntimeError(f"{case}: EP flux was not produced with natural-month N2")
        if case == "0008-03" and opened["nam"].attrs.get("calibration") != "none":
            raise RuntimeError("March NAM must be a direct fixed-EOF projection with no February calibration")
        output = xr.Dataset(
            {
                "partial_o3_du": (("member", "time"), np.asarray(opened["ozone"].partial_o3_du.values)),
                "u60n10": (("member", "time"), np.asarray(opened["dynamics"].u60n10.values)),
                "tmin50": (("member", "time"), np.asarray(opened["dynamics"].tmin50.values)),
                "ep100_upward_40_80n": (("member", "time"), np.asarray(opened["dynamics"].ep100_upward_40_80n.values)),
                "nam": (("member", "time", "plev"), np.asarray(opened["nam"].nam.values)),
                "ao": (("member", "time"), np.asarray(opened["nam"].ao.values)),
                "reference_partial_o3_du": (("time",), np.asarray(verification.o3_reference.values)),
                "reference_u60n10": (("time",), np.asarray(verification.u60n10_reference.values)),
                "reference_tmin50": (("time",), np.asarray(verification.tmin50_reference.values)),
                "reference_ep100_upward_40_80n": (("time",), np.asarray(verification.ep100_reference.values)),
                "reference_nam": (("time", "plev"), np.asarray(verification.nam_reference.values)),
                "reference_ao": (("time",), np.asarray(verification.ao_reference.values)),
            },
            coords={
                "member": members, "time": np.arange(dates.size),
                "date": ("time", dates), "plev": nam_plev,
            },
            attrs={
                "product_version": PRODUCT_VERSION, "case": case, "member_count": 30,
                "ozone_method": "exact 30--70 hPa; 60--90N cosine mean",
                "epflux_method": "natural-month N2; do_ubar=True; w=None; wave=-1",
                "nam_method": "direct fixed LONGRUN EOF; no empirical calibration",
                "ao_definition": "NAM at exactly 1000 hPa",
                "display_start_date": f"{int(display_dates[0]):08d}",
                "display_end_date": f"{int(display_dates[-1]):08d}",
                "display_day_count": int(display_dates.size),
                "display_window_complete": "True",
            },
        )
        output.plev.attrs.update(units="hPa", positive="down")
        for variable in output.data_vars:
            time_axis = output[variable].dims.index("time")
            require_finite_display_values(
                f"{case} canonical case {variable}", output[variable].values,
                display_indices, time_axis=time_axis,
            )
    finally:
        for dataset in opened.values():
            dataset.close()
        verification.close()
    write_netcdf_atomic(
        output, product_path("cases", f"hindcast_{case}.nc"),
        required_vars={
            "partial_o3_du": ("member", "time"), "u60n10": ("member", "time"),
            "tmin50": ("member", "time"), "ep100_upward_40_80n": ("member", "time"),
            "nam": ("member", "time", "plev"), "ao": ("member", "time"),
            "reference_partial_o3_du": ("time",), "reference_u60n10": ("time",),
            "reference_tmin50": ("time",), "reference_ep100_upward_40_80n": ("time",),
            "reference_nam": ("time", "plev"), "reference_ao": ("time",),
        }, required_coords=("member", "date", "plev"), exact_sizes={"member": 30},
        required_attrs={
            "display_end_date": "00080531", "display_window_complete": "True",
            "display_day_count": int(display_dates.size),
        },
        overwrite=OVERWRITE,
    )


## January canonical case

Combines only schema-validated staging products; no legacy public product is changed.

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
assemble_case("0008-01")


## February canonical case

Combines only schema-validated staging products; no legacy public product is changed.

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
assemble_case("0008-02")


## March canonical case

The March gate explicitly rejects any NAM product carrying an empirical calibration.

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
assemble_case("0008-03")
